In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve
from xgboost import XGBClassifier

# ===================== Path setup =====================

RESULTS_BASE = "path/to/outputs"
os.makedirs(RESULTS_BASE, exist_ok=True)

input_path = "path/to/inputs"

# Metadata / Bracken paths
TCGA_META_4COLS = os.path.join(input_path, "Metadata", "TCGA_metadata_4cols.tsv")
NONTCGA_META_4COLS = os.path.join(input_path, "Metadata", "nonTCGA_metadata_4cols.tsv")
TCGA_BRACKEN_DIR = os.path.join(input_path, "TCGA_bracken")
NONTCGA_BRACKEN_DIR = os.path.join(input_path, "Non_TCGA_bracken")

# Metacontam output folders
HMS_metacontam_dir = os.path.join(input_path, "Metacontam_output_folder", "metacontam_HMS_out")
WUSM_metacontam_dir = os.path.join(input_path, "Metacontam_output_folder", "metacontam_WUSM")
BCM_metacontam_dir = os.path.join(input_path, "Metacontam_output_folder", "metacontam_BCM_out")
nonTCGA_metacontam_dir = os.path.join(
    input_path, "Metacontam_output_folder", "metacontam_CRC_non_TCGA"
)

# Squeegee output folders
WUSM_squeegee_dir = os.path.join(input_path, "Squeegee_output_folder", "squeegee_WUSM_out")
BCM_squeegee_dir = os.path.join(input_path, "Squeegee_output_folder", "squeegee_BCM_out")
nonTCGA_squeegee_dir = os.path.join(
    input_path, "Squeegee_output_folder", "squeegee_CRC_non_TCGA"
)

# Decontam output folder (text file lists)
DECONTAM_DIR = os.path.join(input_path, "Decontam_output_folder")
DECONTAM_FILENAMES = {
    "HMS": "hms_contaminant_taxids.txt",
    "BCM": "bcm_contaminant_taxids.txt",
    "MDA": "mda_contaminant_taxids.txt",
    "WUSM": "wusm_contaminant_taxids.txt",
    # nonTCGA has no Decontam output file, so we keep it empty
}

# ===================== Utility functions =====================


def load_metacontam_taxids(
    out_dir,
    filename="Final_prediction.txt",
    status_keys=("contamination_status", "status"),
    taxid_keys=("Taxid", "taxid"),
):
    path = os.path.join(out_dir, filename)
    if not os.path.exists(path) and os.path.exists(path + ".gz"):
        path = path + ".gz"
    if not os.path.exists(path):
        raise FileNotFoundError(f"Not found: {path}")

    df = pd.read_csv(path, sep="\t", dtype=str)

    lower_map = {c.lower(): c for c in df.columns}
    status_col = next((lower_map[k.lower()] for k in status_keys if k.lower() in lower_map), None)
    taxid_col = next((lower_map[k.lower()] for k in taxid_keys if k.lower() in lower_map), None)

    if status_col is None or taxid_col is None:
        raise KeyError(
            f"Required columns not found in {path}. "
            f"status in {status_keys}, taxid in {taxid_keys}"
        )

    contam_mask = df[status_col].str.strip().str.lower().eq("contaminant")
    taxids = (
        df.loc[contam_mask, taxid_col]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
        .tolist()
    )
    return taxids


def load_squeegee_taxids(input_dir, filename="final_predictions.txt"):
    path = os.path.join(input_dir, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Not found: {path}")

    contam_taxid = []
    with open(path) as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            contam_taxid.append(line.split("\t")[0].strip())
    return contam_taxid


def load_decontam_taxids_from_file(path):
    """Load a plain text file: one taxid per line."""
    if not os.path.exists(path):
        # If missing, treat as empty list (some centers may not have files)
        print(f"[Decontam] Warning: Not found {path}. Using empty list.")
        return []

    taxids = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            taxids.append(line)
    return taxids


def build_decontam_map(base_dir=DECONTAM_DIR, filenames=DECONTAM_FILENAMES):
    """Center name (uppercase key) -> list of contaminant taxids."""
    m = {}
    for center, fn in filenames.items():
        path = os.path.join(base_dir, fn)
        m[center] = load_decontam_taxids_from_file(path)

    # nonTCGA has no Decontam results
    m["nonTCGA"] = []
    return m


def parse_bracken_series(bracken_file):
    df = pd.read_csv(bracken_file, sep="\t", dtype={"taxonomy_id": str})

    cols_lower = {c.lower(): c for c in df.columns}
    tax_col = cols_lower.get("taxonomy_id", "taxonomy_id")

    count_col = None
    for cand in [
        "new_est_reads",
        "new_est_read",
        "new_estimated_reads",
        "kraken_assigned_seqs",
        "kraken_assigned_reads",
    ]:
        if cand in cols_lower:
            count_col = cols_lower[cand]
            break

    if count_col is None:
        num_cols = [
            c
            for c in df.columns
            if c != tax_col and pd.api.types.is_numeric_dtype(df[c])
        ]
        if not num_cols:
            return pd.Series(dtype=float)
        count_col = num_cols[-1]

    s = df[[tax_col, count_col]].copy()
    s[count_col] = pd.to_numeric(s[count_col], errors="coerce").fillna(0.0)
    s = s.groupby(tax_col, as_index=True)[count_col].sum()
    s.index = s.index.astype(str)
    return s


def build_matrix_from_dir(bracken_dir, wanted_samples=None):
    files = [f for f in os.listdir(bracken_dir) if f.endswith(".bracken")]
    mat = {}

    for fn in files:
        sid = fn[:-8]  # strip ".bracken"
        if wanted_samples is not None and sid not in wanted_samples:
            continue

        s = parse_bracken_series(os.path.join(bracken_dir, fn))
        if s.empty:
            continue
        mat[sid] = s

    if not mat:
        return pd.DataFrame()

    return pd.DataFrame(mat).fillna(0.0)


def remove_taxid_9606(matrix):
    return matrix[matrix.index != "9606"]


def merge_matrices(M_tcga, M_non):
    all_taxa = sorted(set(M_tcga.index).union(set(M_non.index)))
    M_tcga_merged = M_tcga.reindex(all_taxa, fill_value=0)
    M_non_merged = M_non.reindex(all_taxa, fill_value=0)
    return pd.concat([M_tcga_merged, M_non_merged], axis=1)


def calculate_mean_relative_abundance(M):
    totals = M.sum(axis=0).replace(0, np.nan)
    rel = M.div(totals, axis=1)
    return rel.mean(axis=1).fillna(0.0)


def apply_center_specific_removal(M, sample_to_center, contam_map):
    if M.empty:
        return M

    M = M.copy()
    M.index = M.index.astype(str)

    for center, taxids in (contam_map or {}).items():
        if not taxids:
            continue

        taxids = [str(t) for t in taxids]
        cols = [c for c in M.columns if sample_to_center.get(c) == center]
        if not cols:
            continue

        hit = [t for t in taxids if t in M.index]
        if not hit:
            continue

        M.loc[hit, cols] = 0.0

    return M.loc[M.sum(axis=1) > 0]


def xgb_auc_and_model(train_data, ytr, test_data, yte, best_params):
    model = XGBClassifier(
        objective="binary:logistic",
        learning_rate=best_params["learning_rate"],
        max_depth=best_params["max_depth"],
        n_estimators=best_params["n_estimators"],
        subsample=best_params["subsample"],
        eval_metric="logloss",
        verbosity=0,
    )
    model.fit(train_data.T, ytr)
    scores = model.predict_proba(test_data.T)[:, 1]
    auc = roc_auc_score(yte, scores)
    return auc, scores, model


In [ ]:
# ===================== Data generating (each tissue) =====================
def make_mats_for_tissue(tissue, use_external_nonTCGA=False,
                         metacontam_map=None, squeegee_map=None, decontam_map=None):
    """tissue: 'colon' | 'esophagus' | 'stomach'"""
    tcga = pd.read_csv(TCGA_META_4COLS, sep="\t", header=None,
                       names=["sample_id", "tissue", "center", "tumor_normal"], dtype=str)
    tcga = tcga[tcga["tissue"] == tissue].copy()

    # 존재하는 bracken만
    tcga_exist = {fn[:-8] for fn in os.listdir(TCGA_BRACKEN_DIR) if fn.endswith(".bracken")}
    tcga = tcga[tcga["sample_id"].isin(tcga_exist)].copy()

    M_tcga = build_matrix_from_dir(TCGA_BRACKEN_DIR, set(tcga["sample_id"]))

    if use_external_nonTCGA:
        non = pd.read_csv(NONTCGA_META_4COLS, sep="\t", header=None,
                          names=["sample_id", "tissue", "center", "tumor_normal"], dtype=str)
        non = non[non["tissue"] == tissue].copy()
        non_exist = {fn[:-8] for fn in os.listdir(NONTCGA_BRACKEN_DIR) if fn.endswith(".bracken")}
        non = non[non["sample_id"].isin(non_exist)].copy()
        M_non = build_matrix_from_dir(NONTCGA_BRACKEN_DIR, set(non["sample_id"]))
        M_all = remove_taxid_9606(merge_matrices(M_tcga, M_non))
    else:
        M_all = remove_taxid_9606(M_tcga)

    sample_to_center = dict(zip(tcga["sample_id"], tcga["center"]))
    if use_external_nonTCGA:
        sample_to_center.update(dict(zip(non["sample_id"], non["center"])))

    # contamination removal
    M_original   = M_all
    M_metacontam = apply_center_specific_removal(M_all, sample_to_center, metacontam_map)
    M_squeegee   = apply_center_specific_removal(M_all, sample_to_center, squeegee_map)
    M_decontam   = apply_center_specific_removal(M_all, sample_to_center, decontam_map)

    mats = {}
    if use_external_nonTCGA:
        Xtr_idx = list(tcga["sample_id"])
        Xte_idx = list(non["sample_id"])
        mats = {
            "original":   (M_original[Xtr_idx],   M_original[Xte_idx]),
            "metacontam": (M_metacontam[Xtr_idx], M_metacontam[Xte_idx]),
            "squeegee":   (M_squeegee[Xtr_idx],   M_squeegee[Xte_idx]),
            "decontam":   (M_decontam[Xtr_idx],   M_decontam[Xte_idx]),
        }
        ytr = tcga["tumor_normal"].map({"Normal": 0, "Tumor": 1}).astype(int).values
        yte =   non["tumor_normal"].map({"Normal": 0, "Tumor": 1}).astype(int).values
        return mats, ytr, yte
    else:
        # Only TCGA, no test set
        X_idx = list(tcga["sample_id"])
        mats = {
            "original":   (M_original[X_idx],   None),
            "metacontam": (M_metacontam[X_idx], None),
            "squeegee":   (M_squeegee[X_idx],   None),
            "decontam":   (M_decontam[X_idx],   None),
        }
        y = tcga["tumor_normal"].map({"Normal": 0, "Tumor": 1}).astype(int).values
        return mats, y, None


# ===================== 그리드 탐색 & 플롯 저장 =====================
def eval_and_plot_for_tissue(tissue, use_external_nonTCGA=False, top_ns=None,
                             tools=("original","metacontam","squeegee","decontam")):
    if top_ns is None:
        top_ns = list(range(100, 3100, 100))

    # contam 리스트 로드
    metacontam_map = {
        "HMS":     load_metacontam_taxids(HMS_metacontam_dir),
        "BCM":     load_metacontam_taxids(BCM_metacontam_dir),
        "MDA":     [],
        "WUSM":    load_metacontam_taxids(WUSM_metacontam_dir),
        "nonTCGA": load_metacontam_taxids(nonTCGA_metacontam_dir),
    }
    squeegee_map = {
        "HMS":     [],
        "BCM":     load_squeegee_taxids(BCM_squeegee_dir),
        "MDA":     [],
        "WUSM":    load_squeegee_taxids(WUSM_squeegee_dir),
        "nonTCGA": load_squeegee_taxids(nonTCGA_squeegee_dir),
    }
    decontam_map = build_decontam_map(DECONTAM_DIR, DECONTAM_FILENAMES)

    mats, y_train_or_all, y_test = make_mats_for_tissue(
        tissue=tissue,
        use_external_nonTCGA=use_external_nonTCGA,
        metacontam_map=metacontam_map,
        squeegee_map=squeegee_map,
        decontam_map=decontam_map
    )

    out_dir = os.path.join(RESULTS_BASE, tissue)
    os.makedirs(out_dir, exist_ok=True)

    print(f"\n=== Tissue: {tissue} | External test: {use_external_nonTCGA} | Tools: {tools} ===")
    results = {tool: [] for tool in tools}
    best_config = {}

    # ---- Grid Search over Top-N ----
    for top_n in top_ns:
        for key in tools:
            Mtr_cnt, Mte_cnt = mats[key]
            if Mtr_cnt is None or Mtr_cnt.empty:
                results[key].append((top_n, None, np.nan, np.nan))
                continue

            mean_rel_abundance = calculate_mean_relative_abundance(Mtr_cnt)
            top_taxa = mean_rel_abundance.sort_values(ascending=False).head(top_n).index
            tr_cnt_sel = Mtr_cnt.loc[top_taxa].fillna(0.0)

            grid = GridSearchCV(
                estimator=XGBClassifier(eval_metric="logloss", verbosity=0),
                param_grid={
                    'max_depth': [3, 5],
                    'learning_rate': [0.1],
                    'n_estimators': [100, 300],
                    'subsample': [1],
                },
                cv=5,
                scoring='roc_auc',
                verbose=0
            )
            grid.fit(tr_cnt_sel.T, y_train_or_all)

            best_params = grid.best_params_
            best_auc_train = grid.best_score_

            if use_external_nonTCGA and Mte_cnt is not None:
                te_cnt_sel = Mte_cnt.loc[top_taxa].fillna(0.0)
                auc_test, _, _ = xgb_auc_and_model(
                    tr_cnt_sel, y_train_or_all, te_cnt_sel, y_test, best_params
                )
            else:
                auc_test = best_auc_train

            results[key].append((top_n, best_params, best_auc_train, auc_test))

    # ---- Save TSV & AUC vs TopN plot ----
    color_map = {
        'original':  '#2ca02c',
        'squeegee':  '#1f77b4',
        'metacontam':'#ff7f0e',
        'decontam':  '#ff69b4'
    }

    plt.figure(figsize=(10, 6))
    for key in tools:
        color = color_map.get(key, None)
        df_res = pd.DataFrame(
            results[key],
            columns=["top_n", "params", "train_auc", "test_auc"]
        ).sort_values("top_n")
        df_res.to_csv(os.path.join(out_dir, f"{key}_grid_results.tsv"), sep="\t", index=False)

        df_nonnull = df_res.dropna(subset=["params"])
        if df_nonnull.empty:
            best_config[key] = {
                "top_n": None, "params": None,
                "train_auc": np.nan, "test_auc": np.nan
            }
        else:
            best_idx = df_nonnull["train_auc"].idxmax()
            best_config[key] = df_nonnull.loc[best_idx].to_dict()

        plt.plot(df_res["top_n"], df_res["test_auc"],
                 marker='o', label=f"{key} Test/OOF AUC", color=color)
        plt.plot(df_res["top_n"], df_res["train_auc"],
                 marker='x', linestyle='--', label=f"{key} Train(CV) AUC", color=color)

    plt.xlabel("Top N Species")
    plt.ylabel("AUC")
    plt.title(f"{tissue}: AUC vs Top N Species")

    handles, labels = plt.gca().get_legend_handles_labels()
    order_priority = {'original': 0, 'decontam': 1, 'squeegee': 2, 'metacontam': 3}

    def _key(i):
        tool = labels[i].split()[0].lower()
        sub = 0 if 'Test' in labels[i] else 1
        return (order_priority.get(tool, 99), sub)

    idx = sorted(range(len(labels)), key=_key)
    handles = [handles[i] for i in idx]
    labels  = [labels[i] for i in idx]
    plt.legend(handles, labels)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "AUC_vs_TopN.png"), dpi=300)
    plt.close()

    # ---- ROC (external for colon, OOF for others) ----
    default_order = ["original", "squeegee", "metacontam", "decontam"]
    plot_order = [k for k in default_order if k in tools]

    plt.figure(figsize=(5, 5))
    any_plotted = False

    for key in plot_order:
        cfg = best_config.get(key)
        if not cfg or not cfg["params"] or cfg["top_n"] is None:
            print(f"Skip {key} (no best config)")
            continue

        top_n = int(cfg["top_n"])
        Mtr_cnt, Mte_cnt = mats[key]

        if Mtr_cnt is None or Mtr_cnt.empty:
            print(f"Skip {key} (no matrix)")
            continue

        mean_rel_abundance = calculate_mean_relative_abundance(Mtr_cnt)
        top_taxa = mean_rel_abundance.sort_values(ascending=False).head(top_n).index
        tr_cnt_sel = Mtr_cnt.loc[top_taxa].fillna(0.0)

        # --- External test set (colon) ---
        if use_external_nonTCGA and (Mte_cnt is not None):
            te_cnt_sel = Mte_cnt.loc[top_taxa].fillna(0.0)
            _, scores, _ = xgb_auc_and_model(
                tr_cnt_sel, y_train_or_all, te_cnt_sel, y_test, cfg["params"]
            )
            fpr, tpr, _ = roc_curve(y_test, scores)
            auc_val = roc_auc_score(y_test, scores)

        # --- OOF ROC (esophagus, stomach) ---
        else:
            skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
            oof_scores = np.zeros(len(y_train_or_all), dtype=float)

            for tr_idx, va_idx in skf.split(tr_cnt_sel.T, y_train_or_all):
                X_tr = tr_cnt_sel.T.iloc[tr_idx, :]
                y_tr = y_train_or_all[tr_idx]
                X_va = tr_cnt_sel.T.iloc[va_idx, :]

                model = XGBClassifier(
                    objective='binary:logistic',
                    learning_rate=cfg["params"]['learning_rate'],
                    max_depth=cfg["params"]['max_depth'],
                    n_estimators=cfg["params"]['n_estimators'],
                    subsample=cfg["params"]['subsample'],
                    eval_metric="logloss",
                    verbosity=0
                )
                model.fit(X_tr, y_tr)
                oof_scores[va_idx] = model.predict_proba(X_va)[:, 1]

            fpr, tpr, _ = roc_curve(y_train_or_all, oof_scores)
            auc_val = roc_auc_score(y_train_or_all, oof_scores)

        plt.plot(
            fpr, tpr,
            color=color_map.get(key),
            label=f"{key} (AUC={auc_val:.3f})",
            linewidth=2
        )
        any_plotted = True

    plt.plot([0, 1], [0, 1], 'k--', lw=1)
    plt.xlabel("False Positive Rate", fontsize=12)
    plt.ylabel("True Positive Rate", fontsize=12)
    plt.title(f"{tissue}: ROC Curves (Best Configurations)", fontsize=12)

    handles, labels = plt.gca().get_legend_handles_labels()
    order_priority = {'original': 0, 'decontam': 1, 'squeegee': 2, 'metacontam': 3}

    def _key_roc(i):
        tool = labels[i].split()[0].lower()
        return order_priority.get(tool, 99)

    idx = sorted(range(len(labels)), key=_key_roc)
    handles = [handles[i] for i in idx]
    labels  = [labels[i] for i in idx]

    plt.legend(handles, labels, loc='lower right', fontsize=12)
    plt.grid(False)
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.tight_layout()

    pdf_path = os.path.join(out_dir, "ROC_curves_best.pdf")
    if any_plotted:
        plt.savefig(pdf_path, dpi=300, format='pdf')
    plt.close()

    print(f"Saved ROC to {pdf_path}")

    return best_config


In [ ]:
# ===================== Run: colon / esophagus / stomach =====================

best_cfgs_colon = eval_and_plot_for_tissue(
    "colon",
    use_external_nonTCGA=True,  # External non-TCGA validation set exists
    tools=("original", "metacontam", "squeegee", "decontam"),
)

best_cfgs_esophagus = eval_and_plot_for_tissue(
    "esophagus",
    use_external_nonTCGA=False,  # TCGA 5-fold CV
    tools=("original", "metacontam", "squeegee", "decontam"),
)

best_cfgs_stomach = eval_and_plot_for_tissue(
    "stomach",
    use_external_nonTCGA=False,  # TCGA 5-fold CV
    tools=("original", "metacontam", "decontam"),
)

print("\n=== Best configs summary ===")
print("colon:", best_cfgs_colon)
print("esophagus:", best_cfgs_esophagus)
print("stomach:", best_cfgs_stomach)

# Keep a stable tool order for colon (useful for consistent reporting/plotting)
_tool_order = ("original", "decontam", "squeegee", "metacontam")
best_cfgs_colon_ordered = {k: best_cfgs_colon[k] for k in _tool_order if k in best_cfgs_colon}
for k, v in best_cfgs_colon.items():
    if k not in best_cfgs_colon_ordered:
        best_cfgs_colon_ordered[k] = v
best_cfgs_colon = best_cfgs_colon_ordered


# ===== Paper-safe confusion matrix + AUC bar (colon; FINAL INTEGRATION with Decontam) =====
import os
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score,
    f1_score, roc_curve, roc_auc_score, precision_recall_curve
)
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier


def _load_decontam_taxids_from_file(path):
    """Load decontam taxids (one per line) from a text file."""
    if not os.path.exists(path):
        print(f"[Decontam] Warning: Not found {path}. Using empty list.")
        return []
    taxids = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                taxids.append(line)
    return taxids


def build_decontam_map(base_dir=DECONTAM_DIR, filenames=DECONTAM_FILENAMES):
    """Build a {center: [taxids]} map from the decontam outputs."""
    m = {}
    for center, fn in filenames.items():
        m[center] = _load_decontam_taxids_from_file(os.path.join(base_dir, fn))
    m["nonTCGA"] = []
    return m


# ---------------------------------------------------------------------
# AUC bar plot (avoid label overlap / auto headroom / optional hard cap)
#  → Order: Original, Decontam, Squeegee, Metacontam
# ---------------------------------------------------------------------
def save_auc_barplot(best_cfgs, tissue_name, top_pad=0.20, hard_cap=None,
                     text_offset=10, min_headroom=0.03,
                     eval_label='Test AUC'):

    desired_order = ['Original', 'Decontam', 'Squeegee', 'Metacontam']
    all_tools = list(best_cfgs.keys())
    tools = [t for t in desired_order if t in best_cfgs] + [t for t in all_tools if t not in desired_order]

    train_aucs = [np.nan_to_num(best_cfgs[t].get('train_auc', np.nan), nan=0.0) for t in tools]
    test_aucs  = [np.nan_to_num(best_cfgs[t].get('test_auc',  np.nan), nan=0.0) for t in tools]

    x = np.arange(len(tools))
    width = 0.35
    TRAIN_COLOR = "#7f7f7f"
    TEST_COLOR  = "#9467bd"

    fig, ax = plt.subplots(figsize=(7, 5))
    rects1 = ax.bar(x - width / 2, train_aucs, width, label='Train (CV) AUC', color=TRAIN_COLOR)
    rects2 = ax.bar(x + width / 2, test_aucs,  width, label=eval_label, color=TEST_COLOR)

    ymax = float(np.nanmax(train_aucs + test_aucs)) if len(tools) else 1.0
    upper = ymax + top_pad
    if upper - ymax < min_headroom:
        upper = ymax + min_headroom
    if hard_cap is not None:
        upper = min(hard_cap, upper)
    ax.set_ylim(0, upper)

    ax.set_ylabel('AUC Score')
    ax.set_title(f'{tissue_name}: Best Configurations — Train (CV) vs {eval_label}')
    ax.set_xticks(x)
    ax.set_xticklabels(tools)
    ax.legend(loc='upper right', frameon=True, fancybox=True, framealpha=0.9, borderpad=0.6, labelspacing=0.4)

    def autolabel(rects):
        for rect in rects:
            h = rect.get_height()
            ax.annotate(f'{h:.3f}', xy=(rect.get_x() + rect.get_width() / 2, h),
                        xytext=(0, text_offset), textcoords="offset points",
                        ha='center', va='bottom', fontsize=15, clip_on=False)

    autolabel(rects1)
    autolabel(rects2)

    plt.xticks(fontsize=17)
    plt.yticks(fontsize=17)
    fig.tight_layout()

    tissue_dir = os.path.join(RESULTS_BASE, tissue_name)
    os.makedirs(tissue_dir, exist_ok=True)
    output_file = os.path.join(tissue_dir, f"train_test_auc_comparison_bar_{tissue_name}.pdf")

    fig.set_size_inches(5, 5)
    fig.savefig(output_file, dpi=300, format='pdf', bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f"Saved bar plot: {output_file}")


# ---------------------------------------------------------------------
# Confusion matrix plotter
# ---------------------------------------------------------------------
def plot_confusion_matrix_square(cm, classes=('Normal', 'Tumor'), title='Confusion Matrix',
                                 cmap='Blues', out_path=None, show=False):
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(cm, interpolation='nearest', cmap=cmap)
    ax.set_title(title, fontsize=13)
    ticks = np.arange(len(classes))
    ax.set_xticks(ticks)
    ax.set_xticklabels(classes, fontsize=11)
    ax.set_yticks(ticks)
    ax.set_yticklabels(classes, fontsize=11)
    ax.set_ylabel('True label', fontsize=12)
    ax.set_xlabel('Predicted label', fontsize=12)
    ax.set_ylim(len(classes) - 0.5, -0.5)

    row_sums = cm.sum(axis=1, keepdims=True).astype(float)
    with np.errstate(invalid='ignore', divide='ignore'):
        cm_pct = np.where(row_sums > 0, cm / row_sums, 0.0)

    is_int = np.issubdtype(cm.dtype, np.integer)
    thresh = cm.max() / 2.0 if cm.max() > 0 else 0.5
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            txt = f"{int(cm[i, j])}\n({cm_pct[i, j] * 100:.1f}%)" if is_int else f"{cm_pct[i, j] * 100:.1f}%"
            ax.text(j, i, txt,
                    ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black",
                    fontsize=11)

    fig.tight_layout()
    if out_path:
        fig.savefig(out_path, dpi=300, bbox_inches='tight', format=out_path.split('.')[-1])
    if show:
        plt.show()
    plt.close(fig)


# ===================== Show only ROC_curves_best.pdf (colon / esophagus / stomach) =====================
def _show_pdf_inline(pdf_path, title=None, zoom=2.0):
    """
    Render a PDF (all pages) and display inline.
    Requires PyMuPDF (fitz): pip install pymupdf
    """
    import fitz  # PyMuPDF

    doc = fitz.open(pdf_path)
    for page_idx in range(len(doc)):
        page = doc.load_page(page_idx)
        mat = fitz.Matrix(zoom, zoom)
        pix = page.get_pixmap(matrix=mat, alpha=False)
        img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)

        plt.figure(figsize=(8, 8))
        if title:
            plt.title(title if len(doc) == 1 else f"{title} (page {page_idx + 1}/{len(doc)})")
        plt.imshow(img)
        plt.axis("off")
        plt.show()
    doc.close()


for tissue in ("colon", "esophagus", "stomach"):
    pdf_path = os.path.join(RESULTS_BASE, tissue, "ROC_curves_best.pdf")
    if os.path.exists(pdf_path):
        _show_pdf_inline(pdf_path, title=f"{tissue}: ROC_curves_best.pdf")
    else:
        print(f"[ROC] Not found: {pdf_path}")


In [ ]:
# ---------------------------------------------------------------------
# XGB factory: guard against non-dict params + fill reasonable defaults
# ---------------------------------------------------------------------
def _xgb_from_params(params, random_state=42):
    if not isinstance(params, dict):
        params = {}

    return XGBClassifier(
        objective="binary:logistic",
        learning_rate=params.get("learning_rate", 0.1),
        max_depth=params.get("max_depth", 6),
        n_estimators=params.get("n_estimators", 200),
        subsample=params.get("subsample", 0.8),
        colsample_bytree=params.get("colsample_bytree", 1.0),
        reg_lambda=params.get("reg_lambda", 1.0),
        reg_alpha=params.get("reg_alpha", 0.0),
        eval_metric="logloss",
        verbosity=0,
        random_state=random_state,
        n_jobs=-1,
    )


# ---------------------------------------------------------------------
# Pick a decision threshold from scores
# mode: 'youden' | 'pr_f1' | 'rec@α' | 'prec@α' | 'fixed'
# ---------------------------------------------------------------------
def _pick_threshold_from_scores(y_true, scores, mode="youden", target=0.90, fixed_thr=0.5):
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores, dtype=float)

    if mode == "youden":
        fpr, tpr, thr = roc_curve(y_true, scores)
        m = np.isfinite(thr)
        fpr, tpr, thr = fpr[m], tpr[m], thr[m]
        if len(thr) == 0:
            return 0.5
        j = tpr - fpr
        return float(thr[np.argmax(j)])

    if mode in ("pr_f1", "f1"):
        prec, rec, thr = precision_recall_curve(y_true, scores)
        f1 = 2 * prec * rec / (prec + rec + 1e-12)
        if len(thr) == 0:
            return 0.5
        # precision_recall_curve returns thr of length (len(prec)-1)
        idx = int(np.nanargmax(f1[:-1])) if len(f1) > 1 else 0
        return float(thr[idx])

    if mode.startswith("rec@"):
        try:
            target_rec = float(mode.split("@")[1])
        except Exception:
            target_rec = float(target)

        grid = np.unique(scores)[::-1]
        for t in grid:
            yp = (scores >= t).astype(int)
            tp = np.sum((y_true == 1) & (yp == 1))
            fn = np.sum((y_true == 1) & (yp == 0))
            rec_ = tp / (tp + fn + 1e-12)
            if rec_ >= target_rec:
                return float(t)
        return float(grid[-1]) if len(grid) else 0.5

    if mode.startswith("prec@"):
        try:
            target_prec = float(mode.split("@")[1])
        except Exception:
            target_prec = float(target)

        grid = np.unique(scores)
        chosen = None
        for t in grid:
            yp = (scores >= t).astype(int)
            tp = np.sum((y_true == 1) & (yp == 1))
            fp = np.sum((y_true == 0) & (yp == 1))
            prec_ = tp / (tp + fp + 1e-12)
            if prec_ >= target_prec:
                chosen = float(t)
                break

        if chosen is not None:
            return chosen
        return float(grid[-1]) if len(grid) else 0.5

    if mode == "fixed":
        return float(fixed_thr)

    raise ValueError(f"Unknown threshold mode: {mode}")


# ---------------------------------------------------------------------
# OOF threshold selection (to avoid information leakage)
# ---------------------------------------------------------------------
def _oof_threshold(
    X_cnt_train,
    y_train,
    params,
    mode="youden",
    n_splits=5,
    random_state=42,
    fixed_thr=0.5,
):
    X = X_cnt_train.T
    y = np.asarray(y_train).astype(int)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    oof_scores = np.zeros(len(y), dtype=float)

    for tr_idx, va_idx in skf.split(X, y):
        model = _xgb_from_params(params, random_state=random_state)
        model.fit(X.iloc[tr_idx, :], y[tr_idx])
        oof_scores[va_idx] = model.predict_proba(X.iloc[va_idx, :])[:, 1]

    thr_star = _pick_threshold_from_scores(y, oof_scores, mode=mode, fixed_thr=fixed_thr)
    oof_auc = roc_auc_score(y, oof_scores)
    return thr_star, oof_auc


# ---------------------------------------------------------------------
# Mean relative abundance helper (fallback if external function is missing)
# ---------------------------------------------------------------------
def _mean_rel_abundance(M):
    try:
        return calculate_mean_relative_abundance(M)
    except NameError:
        col_sums = M.sum(axis=0)
        rel = M.div(col_sums.replace(0, np.nan), axis=1).fillna(0.0)
        return rel.mean(axis=1)


# ---------------------------------------------------------------------
# Main: build confusion matrices using best_cfgs (colon)
# - Saves PDFs
# - Also renders plots (show=True)
# ---------------------------------------------------------------------
def confusion_from_best_cfgs_colon(best_cfgs_colon,
                                   threshold_mode='oof_pr_f1',
                                   fixed_thr=0.5, normalize=False):
    tissue = "colon"
    out_dir = os.path.join(RESULTS_BASE, tissue)
    os.makedirs(out_dir, exist_ok=True)

    metacontam_map = {
        "HMS":     load_metacontam_taxids(HMS_metacontam_dir),
        "BCM":     load_metacontam_taxids(BCM_metacontam_dir),
        "MDA":     [],
        "WUSM":    load_metacontam_taxids(WUSM_metacontam_dir),
        "nonTCGA": load_metacontam_taxids(nonTCGA_metacontam_dir),
    }
    squeegee_map = {
        "HMS":     [],
        "BCM":     load_squeegee_taxids(BCM_squeegee_dir),
        "MDA":     [],
        "WUSM":    load_squeegee_taxids(WUSM_squeegee_dir),
        "nonTCGA": load_squeegee_taxids(nonTCGA_squeegee_dir),
    }
    decontam_map = build_decontam_map(DECONTAM_DIR, DECONTAM_FILENAMES)

    mats, ytr, yte = make_mats_for_tissue(
        tissue=tissue, use_external_nonTCGA=True,
        decontam_map=decontam_map,
        squeegee_map=squeegee_map,
        metacontam_map=metacontam_map
    )

    key_map = {
        "original":  "Original",
        "decontam":  "Decontam",
        "squeegee":  "Squeegee",
        "metacontam": "Metacontam",
    }

    order = ["original", "decontam", "squeegee", "metacontam"]

    for key in order:
        # ✅ Only run Metacontam
        if key != "metacontam":
            continue

        if key not in best_cfgs_colon:
            continue

        pretty = key_map[key]
        cfg = best_cfgs_colon[key]
        if (cfg.get("top_n") is None) or (cfg.get("params") is None):
            print(f"[{pretty}] skip: no best cfg")
            continue

        top_n = int(cfg["top_n"])
        params = cfg["params"]
        if not isinstance(params, dict):
            print(f"[{pretty}] params is not dict; using defaults.")
            params = {}

        Mtr_cnt, Mte_cnt = mats.get(key, (None, None))
        if Mtr_cnt is None or Mte_cnt is None or Mtr_cnt.empty or Mte_cnt.empty:
            print(f"[{pretty}] skip: empty matrix")
            continue

        mean_rel = _mean_rel_abundance(Mtr_cnt)
        top_taxa = mean_rel.sort_values(ascending=False).head(top_n).index
        X_tr = Mtr_cnt.reindex(top_taxa).fillna(0.0)
        X_te = Mte_cnt.reindex(top_taxa).fillna(0.0)

        if threshold_mode == 'fixed':
            thr, oof_auc = float(fixed_thr), np.nan
        else:
            mode = threshold_mode[4:] if threshold_mode.startswith('oof_') else threshold_mode
            thr, oof_auc = _oof_threshold(X_tr, ytr, params, mode=mode)

        model = _xgb_from_params(params, random_state=42)
        model.fit(X_tr.T, ytr)
        scores = model.predict_proba(X_te.T)[:, 1]
        test_auc = roc_auc_score(yte, scores)

        y_pred = (scores >= thr).astype(int)
        cm = confusion_matrix(yte, y_pred, labels=[0, 1])
        cm_plot = (np.nan_to_num(cm.astype(float) / cm.sum(axis=1, keepdims=True))
                   if normalize else cm)

        acc  = accuracy_score(yte, y_pred)
        prec = precision_score(yte, y_pred, zero_division=0)
        rec  = recall_score(yte, y_pred, zero_division=0)
        f1   = f1_score(yte, y_pred, zero_division=0)

        print(f"[{pretty}] TopN={top_n}  thr={thr:.3f}  "
              f"TestAUC={test_auc:.3f}  OOF_AUC={oof_auc:.3f}  "
              f"Acc={acc:.3f}  Prec={prec:.3f}  Rec={rec:.3f}  F1={f1:.3f}")

        suffix = f"_topN{top_n}_{threshold_mode}"
        pdf_path = os.path.join(out_dir, f"{key}_confusion_matrix_test_colon{suffix}.pdf")
        plot_confusion_matrix_square(
            cm_plot, classes=('Normal', 'Tumor'),
            title=f"Colon Test — {pretty} (TopN={top_n}, thr={thr:.2f})",
            cmap='Blues', out_path=pdf_path, show=True
        )
        print(f"Saved confusion matrix: {pdf_path}")



In [ ]:
# ---------------------------------------------------------------------
# 예시 best_cfgs_colon (대소문자 혼용 주의)
# ---------------------------------------------------------------------
def _lower_keyed(best_cfgs_colon):
    out = {}
    for k, v in best_cfgs_colon.items():
        out[k.lower()] = v
    return out


if __name__ == "__main__":
    save_auc_barplot(best_cfgs_colon, "colon", top_pad=0.25, hard_cap=None,
                     text_offset=10, eval_label='Test AUC')

best_cfgs_colon_lower = _lower_keyed(best_cfgs_colon)
confusion_from_best_cfgs_colon(best_cfgs_colon_lower, threshold_mode='oof_pr_f1', normalize=False)


# =========================
# SHAP + FI (GridSearch top-N 일치 버전, TSV에서 best_cfgs 자동 복원)
# =========================
import os
import ast
import json
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from sklearn.preprocessing import QuantileTransformer
from xgboost import XGBClassifier

# -------------------------
# 경로 설정
# -------------------------
RESULTS_BASE = "path/to/outputs/results_tissue/"
os.makedirs(RESULTS_BASE, exist_ok=True)

# taxonomy names
NAMES_DMP = "path/to/taxonomy/names.dmp"

# -------------------------
# 재현성 고정
# -------------------------
np.random.seed(0)
random.seed(0)
XGB_KW = dict(
    objective='binary:logistic',
    eval_metric='logloss',
    verbosity=0,
    random_state=0,
    n_jobs=1,
    tree_method='hist',
    predictor='cpu_predictor'
)

# -------------------------
# TSV에서 best_cfgs 복원
# -------------------------
def _safe_parse_params(x):
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return None
    if isinstance(x, dict):
        return x
    s = str(x).strip()
    if not s or s.lower() in ("none", "nan"):
        return None
    try:
        return ast.literal_eval(s)
    except Exception:
        try:
            return json.loads(s.replace("'", '"'))
        except Exception:
            return None


def load_best_cfgs_for_tissue(tissue, tools=None, results_base=RESULTS_BASE):
    if tools is None:
        tools = ("original", "metacontam", "squeegee", "decontam")

    out_dir = os.path.join(results_base, tissue)
    best_cfgs = {}
    for tool in tools:
        tsv_path = os.path.join(out_dir, f"{tool}_grid_results.tsv")
        if not os.path.exists(tsv_path):
            print(f"[load_best_cfgs_for_tissue] Missing file: {tsv_path} -> skip {tool}")
            continue

        df = pd.read_csv(tsv_path, sep="\t", dtype=str)
        for c in ("top_n", "train_auc", "test_auc"):
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce")

        if "params" in df.columns:
            df["params_obj"] = df["params"].apply(_safe_parse_params)
        else:
            df["params_obj"] = None

        valid = df[df["params_obj"].notna() & df["train_auc"].notna()]
        if valid.empty:
            tmp = df[df["train_auc"].notna()]
            if tmp.empty:
                print(f"[load_best_cfgs_for_tissue] No valid rows for {tool} in {tsv_path}")
                continue
            row = tmp.loc[tmp["train_auc"].idxmax()]
            params = _safe_parse_params(row.get("params"))
            if params is None:
                print(f"[load_best_cfgs_for_tissue] No params for best row of {tool} in {tsv_path}")
                continue
        else:
            row = valid.loc[valid["train_auc"].idxmax()]
            params = row["params_obj"]

        top_n_val = row.get("top_n")
        try:
            top_n_val = int(top_n_val)
        except Exception:
            top_n_val = 500

        best_cfgs[tool] = {
            "top_n": int(top_n_val),
            "params": params,
            "train_auc": float(row.get("train_auc")) if not pd.isna(row.get("train_auc")) else float("nan"),
            "test_auc": float(row.get("test_auc")) if not pd.isna(row.get("test_auc")) else float("nan"),
        }

    return best_cfgs


def load_all_best_cfgs(results_base=RESULTS_BASE):
    globals()["best_cfgs_colon"] = load_best_cfgs_for_tissue(
        "colon", tools=("original", "metacontam", "squeegee", "decontam"), results_base=results_base
    )
    globals()["best_cfgs_esophagus"] = load_best_cfgs_for_tissue(
        "esophagus", tools=("original", "metacontam", "squeegee", "decontam"), results_base=results_base
    )
    globals()["best_cfgs_stomach"] = load_best_cfgs_for_tissue(
        "stomach", tools=("original", "metacontam", "decontam"), results_base=results_base
    )
    globals()["best_cfgs"] = globals()["best_cfgs_colon"]


# -------------------------
# SHAP 계산
# -------------------------
def _mean_rel_abundance(M):
    totals = M.sum(axis=0).replace(0, np.nan)
    rel = M.div(totals, axis=1)
    return rel.mean(axis=1).fillna(0.0)


def _select_top_taxa(M_counts, top_n):
    mean_rel = _mean_rel_abundance(M_counts)
    return mean_rel.sort_values(ascending=False).head(int(top_n)).index.astype(str)


def calc_shap(M_counts, ytr, params, top_n=None, ref_M_for_top=None):
    if top_n is not None:
        refM = M_counts if ref_M_for_top is None else ref_M_for_top
        top_taxa = _select_top_taxa(refM, top_n)
        taxa = [t for t in top_taxa if t in M_counts.index.astype(str)]
        M_use = M_counts.loc[taxa].fillna(0.0)
    else:
        M_use = M_counts.fillna(0.0)

    X = M_use.T
    model = XGBClassifier(
        **XGB_KW,
        learning_rate=params['learning_rate'],
        max_depth=params['max_depth'],
        n_estimators=params['n_estimators'],
        subsample=params['subsample']
    )
    model.fit(X, ytr)

    try:
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X)
        shap_values = np.array(shap_values)
    except Exception:
        explainer = shap.TreeExplainer(model)
        sv = explainer(X)
        shap_values = sv.values if hasattr(sv, "values") else sv

    return np.array(shap_values), X


# -------------------------
# names.dmp 로드
# -------------------------
df_names = pd.read_csv(
    NAMES_DMP, sep=r'\t\|\t', engine='python', header=None,
    names=["taxid", "name_txt", "unique_name", "name_class"]
)
df_names["name_class"] = df_names["name_class"].str.rstrip('\t|')
sci_names = df_names[df_names["name_class"] == "scientific name"][["taxid", "name_txt"]].drop_duplicates()
sci_names["taxid"] = sci_names["taxid"].astype(str)


# -------------------------
# 시각화 메인 함수
# -------------------------
def run_shap_and_fi_sidebyside(
    mats,
    ytr,
    best_cfgs,
    sci_names,
    meta_taxids,
    squeegee_taxids,
    tissue_name,
    top_each=6
):
    tissue_dir = os.path.join(RESULTS_BASE, tissue_name)
    os.makedirs(tissue_dir, exist_ok=True)

    top_n_meta = int(best_cfgs['metacontam']['top_n'])
    top_n_orig = int(best_cfgs['original']['top_n'])

    shap_vals_metacontam, X_meta = calc_shap(
        mats['metacontam'][0], ytr, best_cfgs['metacontam']['params'],
        top_n=top_n_meta, ref_M_for_top=mats['metacontam'][0]
    )
    shap_vals_original, X_original = calc_shap(
        mats['original'][0], ytr, best_cfgs['original']['params'],
        top_n=top_n_orig, ref_M_for_top=mats['original'][0]
    )

    orig_cols = X_original.columns.astype(str).str.strip()
    meta_cols = X_meta.columns.astype(str).str.strip()

    meta_set = set(meta_taxids)
    sq_set   = set(squeegee_taxids)

    meta_only = sorted(meta_set - sq_set)
    sq_only   = sorted(sq_set - meta_set)

    taxid_to_name = dict(zip(sci_names['taxid'].astype(str), sci_names['name_txt']))
    shap_abs_mean_orig = np.abs(shap_vals_original).mean(axis=0)
    shap_df = pd.DataFrame({'taxid': orig_cols.values, 'mean_abs_shap': shap_abs_mean_orig})

    shap_df['group'] = np.where(
        shap_df['taxid'].isin(meta_only), 'meta_only',
        np.where(shap_df['taxid'].isin(sq_only), 'squeegee_only', 'other')
    )

    top_meta_only = shap_df[shap_df['group'] == 'meta_only'].nlargest(top_each, 'mean_abs_shap')
    top_sq_only   = shap_df[shap_df['group'] == 'squeegee_only'].nlargest(top_each, 'mean_abs_shap')

    combined_top = pd.concat([top_meta_only, top_sq_only], axis=0)
    combined_labels = [taxid_to_name.get(t, t) for t in combined_top['taxid']]
    top_taxids = combined_top['taxid'].astype(str).tolist()

    idx_original = [orig_cols.get_loc(t) for t in top_taxids]
    meta_cols_set = set(meta_cols)
    idx_metacontam = [meta_cols.get_loc(t) if t in meta_cols_set else None for t in top_taxids]

    fig, axes = plt.subplots(1, 2, figsize=(13, 8), gridspec_kw={'width_ratios': [1, 0.8]})

    fi_original = np.abs(shap_vals_original[:, idx_original]).mean(axis=0)
    x = np.arange(len(combined_top))
    bar_colors = ['#ff7f0e' if grp == 'meta_only' else '#1f77b4' for grp in combined_top['group']]
    axes[0].barh(x, fi_original, height=0.6, color=bar_colors)
    axes[0].set_yticks(x)
    axes[0].set_yticklabels(combined_labels, fontsize=18)
    axes[0].invert_yaxis()
    axes[0].set_xlabel("Mean |SHAP|", fontsize=17)
    axes[0].set_title("Feature Importance (Original only)", fontsize=14)

    qt = QuantileTransformer(output_distribution='uniform')
    for i, idx in enumerate(idx_original):
        feature_values = X_original.iloc[:, idx].values.reshape(-1, 1)
        feature_values_scaled = qt.fit_transform(feature_values).flatten()
        y_jitter = np.random.uniform(-0.2, 0.2, size=len(feature_values_scaled))
        axes[1].scatter(
            shap_vals_original[:, idx],
            i + y_jitter,
            c=feature_values_scaled,
            cmap='RdBu_r',
            alpha=0.6,
            s=20
        )

    axes[1].set_yticks([])
    axes[1].invert_yaxis()
    axes[1].set_xlabel("SHAP value", fontsize=17)
    axes[1].set_title("SHAP Summary", fontsize=14)

    sm = plt.cm.ScalarMappable(cmap='RdBu_r', norm=plt.Normalize(0, 1))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=axes[1])
    cbar.set_label("Feature value", fontsize=12)

    plt.tight_layout()
    output_file = os.path.join(tissue_dir, "fi_left_shap_right_METAonly_top_SQonly_bottom.pdf")
    plt.savefig(output_file, dpi=300, format='pdf')
    plt.show()
    plt.close()
    print(f"Saved combined FI(left) & SHAP(right) plot to {output_file}")


# -------------------------
# 실행부
# -------------------------
if __name__ == "__main__":
    metacontam_map = {
        "HMS":     load_metacontam_taxids(HMS_metacontam_dir),
        "BCM":     load_metacontam_taxids(BCM_metacontam_dir),
        "MDA":     [],
        "WUSM":    load_metacontam_taxids(WUSM_metacontam_dir),
        "nonTCGA": load_metacontam_taxids(nonTCGA_metacontam_dir),
    }
    squeegee_map = {
        "HMS":     [],
        "BCM":     load_squeegee_taxids(BCM_squeegee_dir),
        "MDA":     [],
        "WUSM":    load_squeegee_taxids(WUSM_squeegee_dir),
        "nonTCGA": load_squeegee_taxids(nonTCGA_squeegee_dir),
    }
    decontam_map = build_decontam_map(DECONTAM_DIR, DECONTAM_FILENAMES)

    mats, ytr, _ = make_mats_for_tissue(
        tissue="colon",
        use_external_nonTCGA=True,
        metacontam_map=metacontam_map,
        squeegee_map=squeegee_map,
        decontam_map=decontam_map
    )

    print("[info] Loading best_cfgs_* from TSV grid results ...")
    load_all_best_cfgs(RESULTS_BASE)
    best_cfgs = globals().get('best_cfgs_colon')

    if not best_cfgs:
        raise RuntimeError("Failed to restore best_cfgs_colon from TSV.")

    colon_meta_taxids = metacontam_map.get('BCM', []) + metacontam_map.get('HMS', [])
    colon_squeegee_taxids = squeegee_map.get('BCM', []) + squeegee_map.get('HMS', [])

    run_shap_and_fi_sidebyside(
        mats=mats,
        ytr=ytr,
        best_cfgs=best_cfgs,
        sci_names=sci_names,
        meta_taxids=colon_meta_taxids,
        squeegee_taxids=colon_squeegee_taxids,
        tissue_name="colon",
        top_each=5
    )
